In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))
from configs.config import FAISS_INDEX_DIR, NOTEBOOKS_DIR

# --- Papermill parameters (overwritten at runtime) ---
faiss_index_path     = str(FAISS_INDEX_DIR)
embedding_model      = "all-MiniLM-L6-v2"
llm_model            = "Qwen/Qwen1.5-0.5B"
retriever_k          = 5
temperature          = 0.2
max_new_tokens       = 512
top_k                = 50
similarity_threshold = 0.8
test_csv_path        = str(NOTEBOOKS_DIR / "evaluation" / "test_questions.csv")
n_test_rows          = 2
experiment_name      = "default_eval_run"
run_id               = "default_run_id"

In [ ]:
# Setup MLflow
import mlflow

# Asegurar que estamos en el run correcto sin iniciar uno nuevo
if mlflow.active_run() is None and "run_id" in globals():
    mlflow.start_run(run_id=run_id)

# Log all parameters
mlflow.log_params({
    "faiss_index_path": faiss_index_path,
    "embedding_model":  embedding_model,
    "llm_model":        llm_model,
    "retriever_k":      retriever_k,
    "temperature":      temperature,
    "max_new_tokens":   max_new_tokens,
    "top_k":            top_k,
    "test_csv_path":    test_csv_path,
    "n_test_rows":      n_test_rows
})

In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA

from src.utils import (
    build_embedder,
    extract_answer,
    cos_sim,
    exact_match,
    rouge_l,
)

In [ ]:
def get_embeds(texts: list, embedder) -> list:
    """Embed a list of texts; handles BGE query-prefix via build_embedder."""
    return embedder.embed_documents(texts)

In [ ]:
# build_embedder handles BGE query-prefix automatically
embedder = build_embedder(embedding_model)

db = FAISS.load_local(
    folder_path=faiss_index_path,
    embeddings=embedder,
    allow_dangerous_deserialization=True,
)
retriever = db.as_retriever(search_kwargs={"k": retriever_k})

In [ ]:
# Load LLM
tokenizer = AutoTokenizer.from_pretrained(llm_model)
model     = AutoModelForCausalLM.from_pretrained(llm_model, device_map="auto")
gen_pipe  = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=max_new_tokens,
    do_sample=True,
    top_k=top_k,
    temperature=temperature
)
llm = HuggingFacePipeline(pipeline=gen_pipe)

In [ ]:
# Build RAG QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

In [ ]:
# Load test data
df = pd.read_csv(test_csv_path, nrows=n_test_rows)
df = df.rename(columns={
    'question': 'query',
    'answer': 'ground_truth_answer',
    'evidence': 'ground_truth_context'
})

In [ ]:
metrics = {
    "correctness":  [],   # cosine(gen_answer, gt_answer)
    "relevance":    [],   # cosine(gen_answer, gt_context)
    "faithfulness": [],   # cosine(gen_answer, retrieved_ctx)
    "rouge_l":      [],   # ROUGE-L F1
    "exact_match":  [],   # normalised exact string match
}
recall_hits = {1: 0, 5: 0, 10: 0}

for _, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating"):
    q          = row["query"]
    gt_answer  = str(row["ground_truth_answer"])
    gt_context = str(row["ground_truth_context"])

    out            = qa_chain({"query": q}, return_only_outputs=True)
    gen_ans        = extract_answer(out["result"])
    retrieved_docs = out["source_documents"]
    retrieved_ctx  = " ".join(d.page_content for d in retrieved_docs)

    # --- batch all embeddings needed for this row in two calls ---
    embs_scalar = get_embeds([gen_ans, gt_answer, gt_context, retrieved_ctx], embedder)
    emb_gen, emb_gt_ans, emb_gt_ctx, emb_ret_ctx = embs_scalar

    # pre-embed retrieved docs once for Recall@K
    embs_docs = get_embeds([d.page_content for d in retrieved_docs], embedder)

    # cosine metrics
    metrics["correctness"].append(cos_sim(emb_gen, emb_gt_ans))
    metrics["relevance"].append(cos_sim(emb_gen, emb_gt_ctx))
    metrics["faithfulness"].append(cos_sim(emb_gen, emb_ret_ctx))

    # string metrics
    metrics["rouge_l"].append(rouge_l(gen_ans, gt_answer))
    metrics["exact_match"].append(exact_match(gen_ans, gt_answer))

    # Recall@K — using pre-computed doc embeddings
    for k in (1, 5, 10):
        for emb_doc in embs_docs[:k]:
            if cos_sim(emb_gt_ctx, emb_doc) > similarity_threshold:
                recall_hits[k] += 1
                break

# --- log all metrics to MLflow ---
total = len(df)
for name, vals in metrics.items():
    mlflow.log_metric(f"avg_{name}", float(np.mean(vals)))
    if name not in ("exact_match",):          # std not meaningful for binary
        mlflow.log_metric(f"std_{name}", float(np.std(vals)))

for k, hits in recall_hits.items():
    mlflow.log_metric(f"recall_at_{k}", hits / total)

# --- console summary ---
print(f"{'Metric':<20} {'Mean':>8}")
print("-" * 30)
for name, vals in metrics.items():
    print(f"{name:<20} {np.mean(vals):>8.4f}")
for k, hits in recall_hits.items():
    print(f"recall_at_{k:<11} {hits/total:>8.4f}")

mlflow.end_run()